In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# NOTEBOOK 03 — GOLD LAYER (STAR SCHEMA & ANALYTICS)
# ══════════════════════════════════════════════════════════════════════════════
import logging
import sys
from pyspark.sql import functions as F

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
CATALOG = "retail_data_project"
SILVER  = f"{CATALOG}.02_silver"
GOLD    = f"{CATALOG}.03_gold"

# Source Tables (Silver)
CUST_SILVER = f"{SILVER}.dim_customer_scd2"
PROD_SILVER = f"{SILVER}.dim_product_scd2"
SALE_SILVER = f"{SILVER}.fact_sales_clean"

# Target Tables (Gold - Star Schema)
DIM_CUSTOMER = f"{GOLD}.dim_customer"
DIM_PRODUCT  = f"{GOLD}.dim_product"
DIM_DATE     = f"{GOLD}.dim_date"
FACT_SALES   = f"{GOLD}.fact_sales"

def get_logger(name: str) -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s"))
        logger.addHandler(h)
        logger.setLevel(logging.INFO)
    return logger

logger = get_logger("03_gold_layer")

def bootstrap_gold():
    logger.info(f"Creating Gold Schema: {GOLD}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

# ══════════════════════════════════════════════════════════════════════════════
# 1. DIM_CUSTOMER (Latest Profile)
# ══════════════════════════════════════════════════════════════════════════════
def build_dim_customer():
    logger.info("Building dim_customer...")
    # Filtering for is_current=True ensures the dashboard reflects where customers live NOW
    df = spark.table(CUST_SILVER).filter("is_current = true") \
              .select("customer_id", "name", "city")
    
    df.write.format("delta").mode("overwrite") \
      .option("overwriteSchema", "true") \
      .saveAsTable(DIM_CUSTOMER)

# ══════════════════════════════════════════════════════════════════════════════
# 2. DIM_PRODUCT (Latest Catalog)
# ══════════════════════════════════════════════════════════════════════════════
def build_dim_product():
    logger.info("Building dim_product...")
    df = spark.table(PROD_SILVER).filter("is_current = true") \
              .select("product_id", "product_name", "category")
    
    df.write.format("delta").mode("overwrite") \
      .option("overwriteSchema", "true") \
      .saveAsTable(DIM_PRODUCT)

# ══════════════════════════════════════════════════════════════════════════════
# 3. DIM_DATE (Calendar Dimension for Time Intelligence)
# ══════════════════════════════════════════════════════════════════════════════
def build_dim_date():
    logger.info("Building dim_date...")
    # Get distinct dates from sales to build the calendar
    df_dates = spark.table(SALE_SILVER).select("order_date").distinct()
    
    df = df_dates.withColumn("date_key", F.date_format("order_date", "yyyyMMdd").cast("int")) \
                 .withColumn("full_date", F.col("order_date")) \
                 .withColumn("day", F.dayofmonth("order_date")) \
                 .withColumn("month", F.month("order_date")) \
                 .withColumn("year", F.year("order_date")) \
                 .withColumn("quarter", F.quarter("order_date")) \
                 .withColumn("day_of_week", F.date_format("order_date", "EEEE")) \
                 .withColumn("month_name", F.date_format("order_date", "MMMM"))
    
    df.write.format("delta").mode("overwrite") \
      .option("overwriteSchema", "true") \
      .saveAsTable(DIM_DATE)

# ══════════════════════════════════════════════════════════════════════════════
# 4. FACT_SALES (Core Metrics)
# ══════════════════════════════════════════════════════════════════════════════
def build_fact_sales():
    logger.info("Building fact_sales...")
    # Business Logic: Join keys and metrics only
    df = spark.table(SALE_SILVER) \
              .withColumn("date_key", F.date_format("order_date", "yyyyMMdd").cast("int")) \
              .select("order_id", "customer_id", "product_id", "date_key", "quantity", "total_amount")
    
    # We partition by date_key to optimize Power BI queries for specific time periods
    df.write.format("delta").mode("overwrite") \
      .option("overwriteSchema", "true") \
      .partitionBy("date_key") \
      .saveAsTable(FACT_SALES)

# ══════════════════════════════════════════════════════════════════════════════
# EXECUTION
# ══════════════════════════════════════════════════════════════════════════════
try:
    bootstrap_gold()
    build_dim_customer()
    build_dim_product()
    build_dim_date()
    build_fact_sales()
    logger.info("GOLD LAYER LOADED SUCCESSFULLY ✓")
except Exception as e:
    logger.error(f"GOLD LAYER LOAD FAILED: {e}")
    raise